# traineo1 — Kaggle 2×T4 / P100 Training
Notebook ini menjalankan training Hybrid Mamba-DiT (`train_hybrid_v2.py`) dari branch `kazetts` di Kaggle.

**Urutan cell:**
1. Config & helpers
2. Download dataset dari Kaggle
3. Clone repo (`kazetts` branch)
4. Setup venv + install dependencies
5. Validasi GPU
6. Download teacher + resume checkpoint dari HF
7. Siapkan vocab
8. Merge CSV metadata
9. Prepare dataset
10. Mamba probe + repair
11. Install flash-attn
12. Login W&B
13. Patch trainer (hotfix compat)
14. Distill training
15. Full training
16. Push checkpoint ke HF
17. Inference test

In [ ]:
# ── Cell 1: Config & Helpers ──────────────────────────────────────────────────
import json
import os
import shutil
import subprocess
from pathlib import Path


def require_hardcoded(name: str, value: str) -> str:
    value = (value or "").strip()
    if value:
        return value
    raise ValueError(f"{name} belum diisi. Isi langsung di cell ini.")


# ── API Keys / Tokens (isi di sini) ────────────────────────────────────────
KAGGLE_USERNAME_RAW = ""
KAGGLE_KEY_RAW      = ""
WANDB_API_KEY_RAW   = ""
HF_TOKEN_RAW        = ""

KAGGLE_USERNAME = require_hardcoded("KAGGLE_USERNAME", KAGGLE_USERNAME_RAW)
KAGGLE_KEY      = require_hardcoded("KAGGLE_KEY",      KAGGLE_KEY_RAW)
WANDB_API_KEY   = require_hardcoded("WANDB_API_KEY",   WANDB_API_KEY_RAW)
HF_TOKEN        = require_hardcoded("HF_TOKEN",        HF_TOKEN_RAW)

# ── W&B ─────────────────────────────────────────────────────────────────────
WANDB_ENTITY  = "haidarmuhammaddzaky-institut-teknologi-sepuluh-nopember"
WANDB_PROJECT = "kaceveozon"

# ── Repo ─────────────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/AneKazek/malesbgt.git"
REPO_BRANCH = "kazetts"

# ── Kaggle Dataset ───────────────────────────────────────────────────────────
KAGGLE_DATASET        = "benedictusryugunawan/tts-indo"
KAGGLE_DATASET_SUBDIR = "data"
USE_METADATA_INDSP    = False

# ── Teacher checkpoint (HF) ──────────────────────────────────────────────────
HF_TEACHER_REPO_ID  = "Eempostor/F5-TTS-INDO-FINETUNE-V2"
HF_TEACHER_FILENAME = "f5_tts_indo_v2.pt"

# ── Resume checkpoint (HF) ── isi repo source-nya di sini ───────────────────
# Kosongkan RESUME_HF_REPO_ID jika mau fresh start (tanpa resume).
RESUME_HF_REPO_ID       = ""   # contoh: "anekazek/kcv-tts-modal-h100-ckpt-20260405-112832"
RESUME_DISTILL_FILENAME = "checkpoints/distill/model_last.pt"
RESUME_FULL_FILENAME    = "checkpoints/full/model_last.pt"

# ── Tambah epoch (kalau resume) ──────────────────────────────────────────────
DISTILL_ADD_EPOCHS               = 10
FULL_ADD_EPOCHS                  = 10
CONTINUE_DISTILL_WARMUP_UPDATES  = 300
CONTINUE_FULL_WARMUP_UPDATES     = 2000

# ── Output HF Repo ───────────────────────────────────────────────────────────
HF_OUTPUT_REPO_PREFIX = "kcv-tts-kaggle-ckpt"
HF_OUTPUT_REPO_OWNER  = "anekazek"

# ── Paths ────────────────────────────────────────────────────────────────────
# Semua data besar (dataset, repo, venv, checkpoint) di /kaggle/temp/
# agar tidak memenuhi /kaggle/working/ (quota 20 GB).
# Hanya output inference yang disimpan ke /kaggle/working/.
WORKDIR      = Path("/kaggle/temp")      # ← semua kerja di sini
INFER_OUT_DIR = Path("/kaggle/working")  # ← hanya hasil inference

DATASET_ROOT     = WORKDIR / "datasets" / "tts_indo"
DATASET_DATA_DIR = DATASET_ROOT / KAGGLE_DATASET_SUBDIR
CSV_1            = DATASET_DATA_DIR / "metadata.csv"
CSV_2            = (DATASET_DATA_DIR / "metadata_indsp.csv") if USE_METADATA_INDSP else None

REPO_DIR    = WORKDIR / "kazetts"
VENV_DIR    = REPO_DIR / ".venv"
VENV_PY     = VENV_DIR / "bin/python"

HF_OUT_DIR           = REPO_DIR / "ckpts/hf/teacher"
TEACHER_CKPT         = HF_OUT_DIR / HF_TEACHER_FILENAME
MERGED_CSV           = REPO_DIR / "data/metadata_merged.csv"
PREPARED_DATASET_DIR = REPO_DIR / "data/datasetku_pinyin"

CKPT_ROOT        = WORKDIR / "ckpts"
DISTILL_TAG      = "distill_datasetku"
FULL_TAG         = "full_datasetku"
DISTILL_DIR      = CKPT_ROOT / DISTILL_TAG
FULL_DIR         = CKPT_ROOT / FULL_TAG
NEXT_RESUME_FILE = WORKDIR / "next_resume_links.txt"

WORKDIR.mkdir(parents=True, exist_ok=True)
INFER_OUT_DIR.mkdir(parents=True, exist_ok=True)


# ── Run helpers ───────────────────────────────────────────────────────────────
def run_cmd(cmd, cwd=None, env=None, timeout=None):
    printable = cmd if isinstance(cmd, str) else " ".join(str(x) for x in cmd)
    print("\n$", printable)
    return subprocess.run(
        cmd, cwd=str(cwd) if cwd else None,
        env=env, check=True, text=True, timeout=timeout,
    )


def run_py(args, cwd=None, env=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd, env=env, timeout=timeout,
    )


print("Config siap.")
print("WORKDIR      :", WORKDIR)
print("INFER_OUT_DIR:", INFER_OUT_DIR)
print("REPO_BRANCH  :", REPO_BRANCH)
print("Dataset root :", DATASET_ROOT)

In [ ]:
# ── Cell 2: Download Dataset dari Kaggle ─────────────────────────────────────
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"
kaggle_json.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
run_cmd(["chmod", "600", str(kaggle_json)])

run_cmd(["python3", "-m", "pip", "install", "-U", "kaggle"])

dl_args = [
    "datasets", "download",
    "-d", KAGGLE_DATASET,
    "-p", str(DATASET_ROOT),
    "--unzip",
]

success = False
for cmd in [
    [shutil.which("kaggle") or "kaggle", *dl_args],
    ["python3", "-m", "kaggle.cli", *dl_args],
]:
    try:
        run_cmd(cmd)
        success = True
        break
    except (subprocess.CalledProcessError, TypeError):
        print("Coba metode lain...")

if not success:
    raise RuntimeError("Gagal download dataset via Kaggle CLI.")

# Auto-resolve CSV jika tidak di path default
if not CSV_1.exists():
    fallback = next(iter(sorted(DATASET_ROOT.glob("**/metadata.csv"))), None)
    if fallback is None:
        raise FileNotFoundError(f"metadata.csv tidak ditemukan di {DATASET_ROOT}")
    CSV_1 = fallback

print("CSV_1 :", CSV_1)
print("CSV_2 :", CSV_2)
run_cmd(["ls", "-lah", str(DATASET_ROOT)])

In [ ]:
# ── Cell 3: Clone Repo (branch kazetts) ──────────────────────────────────────
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd([
    "git", "clone", "--recursive",
    "--branch", REPO_BRANCH,
    REPO_URL, str(REPO_DIR),
])
run_cmd(["ls", "-lah", str(REPO_DIR)])

In [ ]:
# ── Cell 4: Setup venv + Install Dependencies ─────────────────────────────────
if shutil.which("uv") is None:
    run_cmd(["python3", "-m", "pip", "install", "-U", "uv"])

TARGET_PY_MM = "3.11"
TARGET_PY    = Path(f"/usr/bin/python{TARGET_PY_MM}")

run_cmd(["apt-get", "update", "-y"])
run_cmd([
    "apt-get", "install", "-y",
    f"python{TARGET_PY_MM}",
    f"python{TARGET_PY_MM}-venv",
    f"python{TARGET_PY_MM}-dev",
    "build-essential",
])

if not TARGET_PY.exists():
    raise FileNotFoundError(f"Python {TARGET_PY_MM} tidak ditemukan: {TARGET_PY}")

run_cmd(["uv", "venv", "--python", str(TARGET_PY), "--clear", str(VENV_DIR)])
run_cmd([str(VENV_PY), "-c",
    "import sys; print('venv python =', sys.version); "
    "assert sys.version_info[:2] == (3, 11)"
])

run_cmd([
    "uv", "pip", "install", "--python", str(VENV_PY),
    "--upgrade", "pip", "wheel", "setuptools<82",
])

# Pilih requirements profile yang ada
req_candidates = [
    REPO_DIR / "requirements-torch28-cu12-localmatch.txt",
    REPO_DIR / "requirements-kaggle-torch210.txt",
    REPO_DIR / "requirements.txt",
]
REQ_PROFILE = next((p for p in req_candidates if p.exists()), None)
if REQ_PROFILE is None:
    raise FileNotFoundError("Tidak ada file requirements ditemukan di repo.")

print("Requirements profile:", REQ_PROFILE)
run_cmd([
    "uv", "pip", "install", "--python", str(VENV_PY),
    "--index-strategy", "unsafe-best-match",
    "-r", str(REQ_PROFILE),
])

# Paksa torch 2.8.0+cu128
run_cmd([
    "uv", "pip", "install", "--python", str(VENV_PY),
    "--index-url", "https://download.pytorch.org/whl/cu128",
    "--extra-index-url", "https://pypi.org/simple",
    "--index-strategy", "unsafe-best-match",
    "--force-reinstall", "--no-cache-dir",
    "torch==2.8.0+cu128",
    "torchvision==0.23.0+cu128",
    "torchaudio==2.8.0+cu128",
    "nvidia-nccl-cu12==2.27.3",
    "nvidia-nvjitlink-cu12==12.8.93",
])

run_cmd([str(VENV_PY), "-c",
    "import torch, torchaudio, setuptools; "
    "print('torch =', torch.__version__, 'cuda =', torch.version.cuda); "
    "print('torchaudio =', torchaudio.__version__); "
    "print('setuptools =', setuptools.__version__)"
])

In [ ]:
# ── Cell 5: Validasi GPU ──────────────────────────────────────────────────────
run_py([
    "-c",
    "import torch; "
    "print('torch', torch.__version__); "
    "print('cuda_count', torch.cuda.device_count()); "
    "[print(i, torch.cuda.get_device_name(i), 'cc', torch.cuda.get_device_capability(i)) "
    " for i in range(torch.cuda.device_count())]; "
    "assert torch.cuda.device_count() >= 1, 'GPU tidak terdeteksi'",
])

In [ ]:
# ── Cell 6: Download Teacher + Resume Checkpoint dari HF ─────────────────────
run_cmd(["uv", "pip", "install", "--python", str(VENV_PY), "-U", "huggingface_hub"])

HF_OUT_DIR.mkdir(parents=True, exist_ok=True)
DISTILL_DIR.mkdir(parents=True, exist_ok=True)
FULL_DIR.mkdir(parents=True, exist_ok=True)

download_env = os.environ.copy()
download_env["HF_TOKEN"] = HF_TOKEN

has_resume = bool(RESUME_HF_REPO_ID.strip())

download_script = f"""
from pathlib import Path
import os, shutil
from huggingface_hub import hf_hub_download

token          = os.environ['HF_TOKEN']
teacher_repo   = {HF_TEACHER_REPO_ID!r}
teacher_file   = {HF_TEACHER_FILENAME!r}
teacher_out    = Path(r'{TEACHER_CKPT}')
teacher_out.parent.mkdir(parents=True, exist_ok=True)

src = Path(hf_hub_download(repo_id=teacher_repo, filename=teacher_file, token=token))
shutil.copy2(src, teacher_out)
print('teacher:', teacher_out)

has_resume = {has_resume!r}
if has_resume:
    resume_repo   = {RESUME_HF_REPO_ID!r}
    distill_file  = {RESUME_DISTILL_FILENAME!r}
    full_file     = {RESUME_FULL_FILENAME!r}
    distill_out   = Path(r'{DISTILL_DIR / "model_last.pt"}')
    full_out      = Path(r'{FULL_DIR / "model_last.pt"}')
    distill_out.parent.mkdir(parents=True, exist_ok=True)
    full_out.parent.mkdir(parents=True, exist_ok=True)

    src_d = Path(hf_hub_download(repo_id=resume_repo, filename=distill_file, token=token))
    src_f = Path(hf_hub_download(repo_id=resume_repo, filename=full_file, token=token))
    shutil.copy2(src_d, distill_out)
    shutil.copy2(src_f, full_out)
    print('distill:', distill_out)
    print('full   :', full_out)
else:
    print('Tidak ada resume checkpoint — fresh start.')
"""

run_py(["-c", download_script], cwd=REPO_DIR, env=download_env)
run_cmd(["ls", "-lah", str(HF_OUT_DIR)])
if has_resume:
    run_cmd(["ls", "-lah", str(DISTILL_DIR)])
    run_cmd(["ls", "-lah", str(FULL_DIR)])

In [ ]:
# ── Cell 7: Siapkan Vocab (Emilia_ZH_EN_pinyin/vocab.txt) ────────────────────
EMILIA_VOCAB_DIR  = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin"
EMILIA_VOCAB_PATH = EMILIA_VOCAB_DIR / "vocab.txt"
EMILIA_VOCAB_DIR.mkdir(parents=True, exist_ok=True)

if EMILIA_VOCAB_PATH.exists() and EMILIA_VOCAB_PATH.stat().st_size > 0:
    print("Vocab sudah ada:", EMILIA_VOCAB_PATH)
else:
    vocab_script = f"""
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download

target = Path(r'{EMILIA_VOCAB_PATH}')
target.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    ('SWivid/F5-TTS', 'F5TTS_Base/vocab.txt'),
    ('SWivid/F5-TTS', 'F5TTS_v1_Base/vocab.txt'),
]
for repo_id, filename in candidates:
    try:
        src = Path(hf_hub_download(repo_id=repo_id, filename=filename))
        shutil.copy2(src, target)
        print(f'Vocab dari {{repo_id}}/{{filename}} -> {{target}}')
        break
    except Exception as e:
        print(f'Gagal dari {{repo_id}}/{{filename}}: {{e}}')
else:
    raise RuntimeError('Gagal download vocab.')

print('vocab size:', target.stat().st_size)
"""
    run_py(["-c", vocab_script], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(EMILIA_VOCAB_DIR)])

In [ ]:
# ── Cell 8: Merge CSV Metadata ────────────────────────────────────────────────
# prepare_csv_wavs.py mensyaratkan:
#   1. Baris header: audio_file|text
#   2. audio_file berupa absolute path
import csv
import pandas as pd


def _resolve_audio_path(p: str, csv_source: Path) -> str:
    """Kembalikan absolute path dari audio_file.
    Jika sudah absolute, langsung dikembalikan.
    Jika relatif, dicoba resolve terhadap parent folder CSV, lalu DATASET_DATA_DIR.
    """
    p = str(p).strip().replace("\\", "/")
    candidate = Path(p)

    # Sudah absolute dan ada
    if candidate.is_absolute() and candidate.exists():
        return str(candidate)

    # Coba resolve relatif terhadap folder CSV
    rel_to_csv = csv_source.parent / p
    if rel_to_csv.exists():
        return str(rel_to_csv.resolve())

    # Coba resolve relatif terhadap DATASET_DATA_DIR
    rel_to_data = DATASET_DATA_DIR / p
    if rel_to_data.exists():
        return str(rel_to_data.resolve())

    # Fallback: resolve terhadap DATASET_ROOT
    rel_to_root = DATASET_ROOT / p
    if rel_to_root.exists():
        return str(rel_to_root.resolve())

    # Path tidak ditemukan — kembalikan resolved agar error ketauan di step berikutnya
    return str((DATASET_DATA_DIR / p).resolve())


dfs = []
for csv_path in filter(None, [CSV_1, CSV_2]):
    if not csv_path.exists():
        print(f"Skip (tidak ada): {csv_path}")
        continue
    df = pd.read_csv(csv_path, sep="|", header=None, names=["audio_file", "text"])
    df = df.dropna(subset=["audio_file", "text"])
    df["audio_file"] = df["audio_file"].apply(lambda p: _resolve_audio_path(p, csv_path))
    df = df[df["text"].str.strip() != ""]
    dfs.append(df)
    print(f"Loaded {len(df)} rows dari {csv_path.name}")

    # Sanity check: pastikan semua path absolute
    non_abs = df[~df["audio_file"].str.startswith("/")]
    if not non_abs.empty:
        print(f"  WARNING: {len(non_abs)} path masih non-absolute, contoh: {non_abs.iloc[0]['audio_file']}")

if not dfs:
    raise RuntimeError("Tidak ada CSV yang berhasil di-load.")

merged = pd.concat(dfs, ignore_index=True).drop_duplicates(subset=["audio_file"])
MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)

# header=True → tulis baris "audio_file|text" yang diharuskan prepare_csv_wavs.py
merged.to_csv(MERGED_CSV, sep="|", index=False, header=True, quoting=csv.QUOTE_NONE, escapechar="\\")

# Verifikasi
sample = merged["audio_file"].iloc[0]
print(f"\nMerged rows : {len(merged)}")
print(f"Sample path : {sample}")
print(f"Is absolute : {Path(sample).is_absolute()}")
print(f"Saved       : {MERGED_CSV}")

# Preview 3 baris pertama CSV hasil
with open(MERGED_CSV, "r") as f:
    for i, line in enumerate(f):
        if i >= 4:
            break
        print(f"  [{i}] {line.rstrip()}")

In [ ]:
# ── Cell 9: Prepare Dataset ───────────────────────────────────────────────────
PREPARED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

# pastikan f5_tts ter-install di venv
try:
    run_py(["-c", "import f5_tts; print('f5_tts ok')"], cwd=REPO_DIR)
except subprocess.CalledProcessError:
    print("Install f5_tts editable...")
    run_cmd([
        "uv", "pip", "install", "--python", str(VENV_PY), "-e", str(REPO_DIR),
    ], cwd=REPO_DIR)

run_py([
    "src/f5_tts/train/datasets/prepare_csv_wavs.py",
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    "--workers", "4",
], cwd=REPO_DIR)

run_cmd(["ls", "-lah", str(PREPARED_DATASET_DIR)])

In [ ]:
# ── Cell 10: Mamba Probe + Repair ─────────────────────────────────────────────
env = os.environ.copy()
env["WANDB_API_KEY"] = WANDB_API_KEY
env["WANDB_ENTITY"]  = WANDB_ENTITY
env["WANDB_PROJECT"] = WANDB_PROJECT

# setup LD_LIBRARY_PATH dari venv site-packages
site_pkgs = sorted((VENV_DIR / "lib").glob("python*/site-packages"))
if site_pkgs:
    sp = site_pkgs[-1]
    cuda_rel = [
        "nvidia/cublas/lib", "nvidia/cuda_runtime/lib", "nvidia/cudnn/lib",
        "nvidia/cufft/lib",  "nvidia/nccl/lib",         "nvidia/nvjitlink/lib",
    ]
    cuda_libs = [str(sp / r) for r in cuda_rel if (sp / r).exists()]
    if cuda_libs:
        env["LD_LIBRARY_PATH"] = ":".join(cuda_libs + [env.get("LD_LIBRARY_PATH", "")])

env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONFAULTHANDLER"]       = "1"
runtime_env = env.copy()


def run_py_nosync(args, cwd=None, timeout=None):
    return run_cmd(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", *args],
        cwd=cwd, env=runtime_env, timeout=timeout,
    )


MAMBA_PROBE_TIMEOUT = 90

def _probe_mamba(timeout=MAMBA_PROBE_TIMEOUT):
    run_py_nosync(["-u", "-c",
        "import torch; import mamba_ssm; import selective_scan_cuda; "
        "print('mamba probe ok')"
    ], cwd=REPO_DIR, timeout=timeout)


def _repair_mamba():
    print("Repair mamba...")

    # pastikan python headers
    py_mm = subprocess.check_output(
        [str(VENV_PY), "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
        text=True,
    ).strip()
    py_h = Path(f"/usr/include/python{py_mm}/Python.h")
    if not py_h.exists():
        run_cmd(["apt-get", "update", "-y"])
        run_cmd(["apt-get", "install", "-y", f"python{py_mm}-dev", "build-essential"])

    run_cmd([str(VENV_PY), "-m", "pip", "install",
             "--upgrade", "pip", "wheel", "ninja", "setuptools<82"],
             cwd=REPO_DIR, env=runtime_env)

    # coba wheel binary dulu
    wheel_env = runtime_env.copy()
    arch_probe = subprocess.run(
        ["uv", "run", "--no-sync", "--python", str(VENV_PY), "python", "-c",
         "import torch; c=torch.cuda.get_device_capability(0); print(f'{c[0]}.{c[1]}')"],
        cwd=str(REPO_DIR), env=runtime_env, capture_output=True, text=True,
    )
    cuda_arch = arch_probe.stdout.strip().splitlines()[-1] if arch_probe.returncode == 0 else "7.5"
    wheel_env["TORCH_CUDA_ARCH_LIST"] = cuda_arch
    wheel_env["MAX_JOBS"] = "4"

    try:
        run_cmd([
            "uv", "pip", "install", "--python", str(VENV_PY),
            "--force-reinstall", "--no-cache-dir", "--prefer-binary",
            "--no-build-isolation", "--no-deps",
            "causal-conv1d", "mamba-ssm",
        ], cwd=REPO_DIR, env=wheel_env)
        _probe_mamba(timeout=120)
        print("Mamba OK (wheel).")
        return
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("Wheel gagal, build from source...")

    build_env = wheel_env | {"MAMBA_FORCE_BUILD": "TRUE", "CAUSAL_CONV1D_FORCE_BUILD": "TRUE"}
    for pkg in ["causal-conv1d", "mamba-ssm"]:
        run_cmd([
            str(VENV_PY), "-m", "pip", "install",
            "--no-cache-dir", "--no-build-isolation",
            "--force-reinstall", "--no-binary", ":all:", "--no-deps", pkg,
        ], cwd=REPO_DIR, env=build_env)

    _probe_mamba(timeout=240)
    print("Mamba OK (built from source).")


try:
    _probe_mamba()
    print("mamba_mode: enabled (fast-path)")
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    print("Probe gagal — repair...")
    _repair_mamba()
    _probe_mamba(timeout=240)
    print("mamba_mode: enabled (after repair)")

print("Runtime siap.")

In [ ]:
# ── Cell 11: Install Flash-Attention ─────────────────────────────────────────
FLASH_WHL = (
    "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
    "flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl"
)

install_env = runtime_env.copy()
install_env["PIP_NO_DEPS"] = "1"

# uninstall dulu
run_cmd(
    ["uv", "run", "--no-sync", "--python", str(VENV_PY),
     "pip", "uninstall", "-y", "flash-attn", "flash_attn"],
    cwd=REPO_DIR, env=install_env,
)

# install prebuilt wheel tanpa deps
run_cmd(
    ["uv", "run", "--no-sync", "--python", str(VENV_PY),
     "pip", "install", "--no-deps", "--force-reinstall", FLASH_WHL],
    cwd=REPO_DIR, env=install_env,
)

run_py_nosync(
    ["-c", "import flash_attn, torch; "
     "print('flash_attn', flash_attn.__version__, 'torch', torch.__version__)"],
    cwd=REPO_DIR,
)

USE_FLASH_ATTN = True
print("USE_FLASH_ATTN =", USE_FLASH_ATTN)

In [ ]:
# ── Cell 12: Login W&B ────────────────────────────────────────────────────────
wandb_env = runtime_env.copy()
wandb_env["WANDB_API_KEY"] = WANDB_API_KEY
wandb_env["WANDB_ENTITY"]  = WANDB_ENTITY
wandb_env["WANDB_PROJECT"] = WANDB_PROJECT

wandb_smoke = """
import os, random, wandb
wandb.login(key=os.environ["WANDB_API_KEY"])
run = wandb.init(
    entity=os.environ["WANDB_ENTITY"],
    project=os.environ["WANDB_PROJECT"],
    config={"smoke": True},
)
run.log({"smoke_loss": 0.0})
run.finish()
print("wandb sanity done")
""".strip()

run_py(["-c", wandb_smoke], cwd=REPO_DIR, env=wandb_env)

In [ ]:
# ── Cell 13: Hotfix — Patch file Python (trainer.py & train_hybrid_v2.py) ───────
import re

# 1. Patch weights_only=False di model/trainer.py
trainer_py = REPO_DIR / "src/f5_tts/model/trainer.py"
if trainer_py.exists():
    src_trainer = trainer_py.read_text(encoding="utf-8")
    PATCH_OLD = 'checkpoint = torch.load(\n                f"{self.checkpoint_path}/{latest_checkpoint}", weights_only=True, map_location="cpu"\n            )'
    PATCH_NEW = 'checkpoint = torch.load(\n                f"{self.checkpoint_path}/{latest_checkpoint}", weights_only=False, map_location="cpu"\n            )'

    if PATCH_NEW in src_trainer:
        print("✓ Patch weights_only=False sudah ada di trainer.py")
    elif PATCH_OLD in src_trainer:
        trainer_py.write_text(src_trainer.replace(PATCH_OLD, PATCH_NEW, 1), encoding="utf-8")
        print("✓ Patch weights_only=False berhasil diterapkan di trainer.py")
    else:
        new_src, n = re.subn(
            r'(torch\.load\(\s*f"\{self\.checkpoint_path\}/\{latest_checkpoint\}"[^)]*?)weights_only=True',
            r'\1weights_only=False',
            src_trainer,
        )
        if n > 0:
            trainer_py.write_text(new_src, encoding="utf-8")
            print(f"✓ Patch weights_only=False regex berhasil di trainer.py ({n} penggantian).")
        else:
            print("INFO: Pattern torch.load tidak ditemukan di trainer.py")

# 2. Patch script train_hybrid_v2.py
train_py = REPO_DIR / "src/f5_tts/train/train_hybrid_v2.py"
if train_py.exists():
    src_train = train_py.read_text(encoding="utf-8")
    
    # A. Patch Teacher DiT TypeError
    if "valid_args = inspect.signature(teacher_cls.__init__).parameters" not in src_train:
        old_impl = """
    teacher_cfg = model_cfg.model.get("teacher_arch", model_cfg.model.arch)
    teacher_backbone = model_cfg.model.get("teacher_backbone", "DiT")
    teacher_cls = hydra.utils.get_class(f"f5_tts.model.backbones.{teacher_backbone.lower()}.{teacher_backbone}")
    
    teacher = teacher_cls(
        **teacher_cfg,
        text_num_embeds=vocab_size,
        mel_dim=mel_dim
    )"""
        new_impl = """
    import inspect
    teacher_cfg = model_cfg.model.get("teacher_arch", model_cfg.model.arch)
    teacher_cfg = dict(teacher_cfg)
    teacher_backbone = model_cfg.model.get("teacher_backbone", "DiT")
    teacher_cls = hydra.utils.get_class(f"f5_tts.model.backbones.{teacher_backbone.lower()}.{teacher_backbone}")
    
    valid_args = inspect.signature(teacher_cls.__init__).parameters
    teacher_kwargs = {k: v for k, v in teacher_cfg.items() if k in valid_args}
    
    teacher = teacher_cls(
        **teacher_kwargs,
        text_num_embeds=vocab_size,
        mel_dim=mel_dim
    )"""
        old_impl = old_impl.split("\n", 1)[1] if old_impl.startswith("\n") else old_impl
        new_impl = new_impl.split("\n", 1)[1] if new_impl.startswith("\n") else new_impl
        
        if old_impl in src_train:
            src_train = src_train.replace(old_impl, new_impl, 1)
            print("✓ Patch setup_teacher kwargs filter berhasil diterapkan di train_hybrid_v2.py")
        else:
            print("INFO: Patch setup_teacher tidak ter-apply, pattern tidak cocok persis.")
    else:
        print("✓ Patch setup_teacher sudah ada di train_hybrid_v2.py")

    # B. Patch CFM TypeError (teacher_ckpt_path)
    if "valid_cfm_args =" not in src_train:
        old_cfm = """
    cfm_experiment = OmegaConf.to_container(cfg.model.cfm_experiment, resolve=True)
    cfm_experiment["teacher_transformer"] = teacher
    
    model = CFM(
        transformer=student_transformer,
        mel_spec_kwargs=cfg.model.mel_spec,
        vocab_char_map=vocab_char_map,
        **cfm_experiment
    )"""
        new_cfm = """
    cfm_experiment_full = OmegaConf.to_container(cfg.model.cfm_experiment, resolve=True)
    import inspect
    valid_cfm_args = inspect.signature(CFM.__init__).parameters
    cfm_kwargs = {k: v for k, v in cfm_experiment_full.items() if k in valid_cfm_args}
    cfm_kwargs["teacher_transformer"] = teacher
    
    model = CFM(
        transformer=student_transformer,
        mel_spec_kwargs=OmegaConf.to_container(cfg.model.mel_spec, resolve=True) if hasattr(cfg.model, "mel_spec") else dict(),
        vocab_char_map=vocab_char_map,
        **cfm_kwargs
    )"""
        old_cfm = old_cfm.split("\n", 1)[1] if old_cfm.startswith("\n") else old_cfm
        new_cfm = new_cfm.split("\n", 1)[1] if new_cfm.startswith("\n") else new_cfm
        
        if old_cfm in src_train:
            src_train = src_train.replace(old_cfm, new_cfm, 1)
            print("✓ Patch CFM kwargs filter berhasil diterapkan di train_hybrid_v2.py")
        else:
            print("WARNING: Gagal mem-patch CFM di train_hybrid_v2.py secara otomatis, pattern tidak cocok.")
    else:
        print("✓ Patch CFM sudah ada.")
        
    train_py.write_text(src_train, encoding="utf-8")

In [ ]:
# ── Cell 14: Distill Training ─────────────────────────────────────────────────
# Semua hyperparameter training diatur via command di sini —
# tidak ada hardcode batch size di config; ganti sesuai GPU yang tersedia.
import math

NUM_WORKERS = 4

# Cek apakah ada resume checkpoint
distill_resume = DISTILL_DIR / "model_last.pt"
has_distill_resume = distill_resume.exists()

distill_env = runtime_env.copy()
distill_env.update({
    "WANDB_API_KEY": WANDB_API_KEY,
    "WANDB_ENTITY":  WANDB_ENTITY,
    "WANDB_PROJECT": WANDB_PROJECT,
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,roundup_power2_divisions:16",
    "OMP_NUM_THREADS": str(NUM_WORKERS),
    "MKL_NUM_THREADS": str(NUM_WORKERS),
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    "PYTHONFAULTHANDLER": "1",
})

# flash_attn backend jika tersedia
attn_override = []
if USE_FLASH_ATTN:
    try:
        run_py_nosync(["-c", "import flash_attn; print('ok')"], cwd=REPO_DIR, timeout=30)
        attn_override = ["model.arch.attn_backend=flash_attn"]
        print("Using flash_attn backend.")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
        print("flash_attn tidak tersedia, fallback ke torch.")

distill_run_name = "F5TTS_HybridV2_distill_datasetku_kaggle_traineo1"
distill_save_dir = f"../ckpts/{DISTILL_TAG}"

distill_cmd = [
    "uv", "run", "--no-sync", "--python", str(VENV_PY),
    "accelerate", "launch",
    "--num_processes=2",          # 2xT4 di Kaggle; ubah ke 1 jika P100
    "--mixed_precision=bf16",
    "--dynamo_backend=no",
    "src/f5_tts/train/train_hybrid_v2.py",
    "--config-name", "F5TTS_SparseBiMamba_V2",
    "datasets.name=datasetku",
    f"+model.cfm_experiment.teacher_ckpt_path={TEACHER_CKPT}",  # <-- butuh + di sini karena key belum ada
    "++model.cfm_experiment.use_distill=true",
    "++model.cfm_experiment.freeze_student_except_mamba_mixers=true",
    # ── Hyperparameter — UBAH DI SINI ──────────────────────────────────────
    "datasets.batch_size_per_gpu=8000",    # sesuaikan VRAM; T4=8000, A100=38400
    "datasets.max_samples=64",             # sesuaikan VRAM
    f"datasets.num_workers={NUM_WORKERS}",
    "optim.epochs=10",
    f"optim.num_warmup_updates={CONTINUE_DISTILL_WARMUP_UPDATES}",
    "optim.learning_rate=2e-5",
    "optim.grad_accumulation_steps=2",
    "++model.cfm_experiment.lambda_distill_out=0.1",
    "model.arch.checkpoint_activations=True",  # hemat VRAM
    # ── Checkpoint ─────────────────────────────────────────────────────────
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={distill_run_name}",
    f"ckpts.save_dir={distill_save_dir}",
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=200",
    "ckpts.keep_last_n_checkpoints=2",
    "ckpts.log_samples=False",
    *attn_override,
]

run_cmd(distill_cmd, cwd=REPO_DIR, env=distill_env)
run_cmd(["ls", "-lah", str(DISTILL_DIR)])

distill_last = DISTILL_DIR / "model_last.pt"
if not distill_last.exists():
    raise FileNotFoundError(f"Distill checkpoint tidak ditemukan: {distill_last}")

print("Distill selesai:", distill_last)

In [ ]:
# ── Cell 15: Full Training ────────────────────────────────────────────────────
full_env = runtime_env.copy()
full_env.update({
    "WANDB_API_KEY": WANDB_API_KEY,
    "WANDB_ENTITY":  WANDB_ENTITY,
    "WANDB_PROJECT": WANDB_PROJECT,
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,roundup_power2_divisions:16",
    "OMP_NUM_THREADS": str(NUM_WORKERS),
    "MKL_NUM_THREADS": str(NUM_WORKERS),
    "TORCH_ALLOW_TF32_CUBLAS_OVERRIDE": "1",
    "PYTHONFAULTHANDLER": "1",
})

full_run_name = "F5TTS_HybridV2_full_datasetku_kaggle_traineo1"
full_save_dir  = f"../ckpts/{FULL_TAG}"

full_cmd = [
    "uv", "run", "--no-sync", "--python", str(VENV_PY),
    "accelerate", "launch",
    "--num_processes=2",          # 2xT4 di Kaggle; ubah ke 1 jika P100
    "--mixed_precision=bf16",
    "--dynamo_backend=no",
    "src/f5_tts/train/train_hybrid_v2.py",
    "--config-name", "F5TTS_SparseBiMamba_V2",
    "datasets.name=datasetku",
    "++model.cfm_experiment.use_distill=false",
    "++model.cfm_experiment.freeze_student_except_mamba_mixers=false",
    # ── Hyperparameter — UBAH DI SINI ──────────────────────────────────────
    "datasets.batch_size_per_gpu=10000",   # sesuaikan VRAM; T4=10000, A100=51200
    "datasets.max_samples=64",
    f"datasets.num_workers={NUM_WORKERS}",
    "optim.epochs=10",
    f"optim.num_warmup_updates={CONTINUE_FULL_WARMUP_UPDATES}",
    "optim.grad_accumulation_steps=2",
    "model.arch.checkpoint_activations=True",
    # ── Checkpoint ─────────────────────────────────────────────────────────
    "ckpts.logger=wandb",
    f"ckpts.wandb_project={WANDB_PROJECT}",
    f"ckpts.wandb_run_name={full_run_name}",
    f"ckpts.save_dir={full_save_dir}",
    "ckpts.save_per_updates=999999999",
    "ckpts.last_per_updates=999999999",
    "ckpts.keep_last_n_checkpoints=0",
    "ckpts.log_samples=False",
    *attn_override,
]

run_cmd(full_cmd, cwd=REPO_DIR, env=full_env)
run_cmd(["ls", "-lah", str(FULL_DIR)])

full_last = FULL_DIR / "model_last.pt"
if not full_last.exists():
    raise FileNotFoundError(f"Full checkpoint tidak ditemukan: {full_last}")

print("Full training selesai:", full_last)

In [ ]:
# ── Cell 16: Push Checkpoint ke Hugging Face ─────────────────────────────────
from datetime import datetime, timezone
from uuid import uuid4

try:
    from huggingface_hub import HfApi
except ImportError:
    run_cmd(["pip", "install", "-q", "-U", "huggingface_hub"])
    from huggingface_hub import HfApi

api   = HfApi(token=HF_TOKEN)
owner = (HF_OUTPUT_REPO_OWNER or api.whoami(token=HF_TOKEN).get("name", "")).strip()
if not owner:
    raise RuntimeError("Owner HF tidak terdeteksi. Isi HF_OUTPUT_REPO_OWNER.")

timestamp      = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
target_repo_id = None

for _ in range(5):
    candidate = f"{owner}/{HF_OUTPUT_REPO_PREFIX}-{timestamp}-{uuid4().hex[:8]}"
    try:
        api.create_repo(repo_id=candidate, repo_type="model", private=True, exist_ok=False, token=HF_TOKEN)
        target_repo_id = candidate
        break
    except Exception as exc:
        if not any(t in str(exc).lower() for t in ("already exists", "409", "conflict")):
            raise

if target_repo_id is None:
    raise RuntimeError("Gagal membuat repo HF baru.")

print("Target repo:", target_repo_id)

# Upload distill
api.upload_file(
    path_or_fileobj=str(distill_last),
    path_in_repo="checkpoints/distill/model_last.pt",
    repo_id=target_repo_id, repo_type="model", token=HF_TOKEN,
)
print("✓ distill uploaded")

# Upload full
api.upload_file(
    path_or_fileobj=str(full_last),
    path_in_repo="checkpoints/full/model_last.pt",
    repo_id=target_repo_id, repo_type="model", token=HF_TOKEN,
)
print("✓ full uploaded")

repo_url     = f"https://huggingface.co/{target_repo_id}"
distill_url  = f"{repo_url}/blob/main/checkpoints/distill/model_last.pt"
full_url     = f"{repo_url}/blob/main/checkpoints/full/model_last.pt"

# Simpan link untuk run berikutnya
NEXT_RESUME_FILE.write_text(
    "\n".join([
        "Hardcode untuk run berikutnya:",
        f'RESUME_HF_REPO_ID       = "{target_repo_id}"',
        f'RESUME_DISTILL_FILENAME = "checkpoints/distill/model_last.pt"',
        f'RESUME_FULL_FILENAME    = "checkpoints/full/model_last.pt"',
        "",
        f"Repo URL    : {repo_url}",
        f"Distill URL : {distill_url}",
        f"Full URL    : {full_url}",
    ]),
    encoding="utf-8",
)

print("Upload selesai.")
print("Repo URL :", repo_url)
print("Next resume links:", NEXT_RESUME_FILE)

In [ ]:
# ── Cell 17: Inference Test ───────────────────────────────────────────────────
import csv as _csv
from IPython.display import Audio, display

# Pilih checkpoint: full dulu, fallback ke distill
ckpt_file = full_last if ("full_last" in globals() and Path(full_last).exists()) else distill_last
if not Path(ckpt_file).exists():
    raise FileNotFoundError(f"Checkpoint tidak ada: {ckpt_file}")

# Ambil referensi audio dari merged CSV
with open(MERGED_CSV, "r", encoding="utf-8") as f:
    rows = [r for r in _csv.DictReader(f, delimiter="|", fieldnames=["audio_file", "text"])
            if r.get("audio_file", "").strip() and r.get("text", "").strip()]

if not rows:
    raise ValueError(f"metadata kosong: {MERGED_CSV}")

ref_row  = rows[0]
ref_text = ref_row["text"].strip()
gen_text = "halo, selamat datang di kazetts"

# Resolve path audio referensi
def _find_audio(p: str) -> Path:
    for base in [Path(p), DATASET_ROOT / p, REPO_DIR / p, WORKDIR / p]:
        if Path(base).exists():
            return Path(base).resolve()
    raise FileNotFoundError(f"Audio referensi tidak ketemu: {p}")

ref_audio = _find_audio(ref_row["audio_file"])

vocab_file = REPO_DIR / "data" / "Emilia_ZH_EN_pinyin" / "vocab.txt"
if not vocab_file.exists():
    vocab_file = next(
        (p for p in (REPO_DIR / "data").glob("**/vocab.txt")
         if p.is_file() and p.stat().st_size > 0),
        None,
    )
    if vocab_file is None:
        raise FileNotFoundError("vocab.txt tidak ditemukan")

# Output ke /kaggle/working/ agar bisa diakses & didownload dari Kaggle UI
out_wav = INFER_OUT_DIR / "infer_traineo1_test.wav"

infer_script = f"""
from pathlib import Path
from f5_tts.api import F5TTS
import torch

ckpt_file  = Path(r"{ckpt_file}")
vocab_file = Path(r"{vocab_file}")
ref_audio  = Path(r"{ref_audio}")
out_wav    = Path(r"{out_wav}")
ref_text   = {ref_text!r}
gen_text   = {gen_text!r}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device    =", device)
print("ckpt      =", ckpt_file)
print("ref_audio =", ref_audio)
print("gen_text  =", gen_text)

tts = F5TTS(
    model="F5TTS_EarlyBiMamba_v1",
    ckpt_file=str(ckpt_file),
    vocab_file=str(vocab_file),
    use_ema=True,
    device=device,
)

for m in [getattr(tts, "ema_model", None), getattr(tts, "model", None)]:
    if m is not None:
        m.float()

tts.infer(
    ref_file=str(ref_audio),
    ref_text=ref_text,
    gen_text=gen_text,
    file_wave=str(out_wav),
    nfe_step=32,
    speed=1.0,
    seed=42,
)
print("DONE:", out_wav)
"""

run_py(["-c", infer_script], cwd=REPO_DIR, env={**runtime_env, "MPLBACKEND": "Agg"})

print("Saved:", out_wav)
display(Audio(str(out_wav)))

## Notes

- **Platform target**: Kaggle 2×T4 (atau P100 — ubah `--num_processes=2` ke `1` untuk P100)
- **Backbone**: `train_hybrid_v2.py` dari branch `kazetts`
- **Dataset**: `benedictusryugunawan/tts-indo` (Kaggle)
- **Storage layout**:
  - `/kaggle/temp/` → semua file kerja: dataset, repo, venv, checkpoint (tidak persistent setelah session berakhir, tapi bebas dari quota 20 GB `/kaggle/working/`)
  - `/kaggle/working/` → **hanya** output inference (`infer_traineo1_test.wav`) agar bisa didownload dari Kaggle UI
- **Batch size**: Diatur langsung di cell training (cell 14 & 15), bukan di config — sesuaikan dengan VRAM yang tersedia
  - T4 (16 GB): `batch_size_per_gpu=8000–10000`, `max_samples=64`
  - A100/H100: `batch_size_per_gpu=38400–51200`, `max_samples=128`
- **Resume**: Isi `RESUME_HF_REPO_ID` di cell 1 untuk lanjut dari checkpoint sebelumnya; kosongkan untuk fresh start
- **Setelah selesai**: Link repo HF disimpan ke `/kaggle/temp/next_resume_links.txt` — copy ke cell 1 run berikutnya
- **Secret minimal**: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `WANDB_API_KEY`, `HF_TOKEN`